# Notebook 01 — Modelos Baseline

**Tech Challenge FIAP — Fase 2**

Este notebook reproduz os modelos da Fase 1 como **linha de base (baseline)** para comparação com os modelos otimizados via Algoritmo Genético (Notebook 02).

**Modelos treinados:**
- Regressão Logística
- Random Forest
- SVM (kernel RBF)

**Dataset:** Maternal Health Risk — UCI (~790 registros, 6 features clínicas)

**Métrica principal:** F1-macro (problema multiclasse com desbalanceamento moderado)

**Resultado esperado (Fase 1):** Random Forest — F1-macro ~0.90, ROC-AUC ~0.98

## 0. Imports e configurações

In [1]:
import json
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")  # backend não-interativo para execução headless
import matplotlib.pyplot as plt
import pandas as pd

# Garante que src/ esteja no path ao rodar o notebook de dentro de notebooks/
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pipelines.preprocessing import build_preprocessed_splits
from src.models.baseline import build_pipelines, run_cross_validation, run_grid_search, summarize_best_params
from src.evaluation.metrics import evaluate_model, build_comparison_table, plot_feature_importance

print(f"Projeto raiz: {PROJECT_ROOT}")

Projeto raiz: /Users/igornatanael/workspace/tech-challenge-fiap-2


## 1. Pré-processamento e split treino/teste

In [2]:
X_train, X_test, y_train, y_test, scaler = build_preprocessed_splits(
    add_pulse_pressure=False,
    add_age_features=False,
    convert_temp=True,
    scale=True,
)

print(f"\nShapes — X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"Features: {X_train.columns.tolist()}")

PIPELINE DE PRE-PROCESSAMENTO
[load] Dataset carregado: 1014 linhas x 7 colunas
[outliers] Removidas 2 linha(s) com HeartRate < 30 bpm.
[age] Removidos 82 registro(s) com Age > 54 anos.
[dedup] Removidas 140 ocorrencia(s) acima do limite de 3 por grupo. Linhas restantes: 790
[features] BodyTemp convertida de Fahrenheit para Celsius.
[split] Treino: 632 linhas | Teste: 158 linhas (20% teste)
[scaler] StandardScaler aplicado (fit no treino, transform em ambos).
-------------------------------------------------------
Features finais (6): ['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate']
Distribuicao do alvo — treino:
  low risk (0): 277 (43.8%)
  mid risk (1): 198 (31.3%)
  high risk (2): 157 (24.8%)
Distribuicao do alvo — teste:
  low risk (0): 69 (43.7%)
  mid risk (1): 50 (31.6%)
  high risk (2): 39 (24.7%)

Shapes — X_train: (632, 6) | X_test: (158, 6)
Features: ['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate']


## 2. Validação cruzada (k=5) — visão geral dos modelos

In [3]:
pipelines = build_pipelines()

df_cv = run_cross_validation(pipelines, X_train, y_train)

print("\nResultados — Validação Cruzada (k=5):")
df_cv

[CV] Avaliando logistic_regression...
[CV] Avaliando random_forest...


[CV] Avaliando svm...

Resultados — Validação Cruzada (k=5):


,f1_macro_media,f1_macro_std,recall_macro_media,recall_macro_std,accuracy_media,accuracy_std
modelo,,,,,,
logistic_regression,0.5883,0.0331,0.5924,0.0349,0.6013,0.0327
random_forest,0.7872,0.0169,0.7852,0.0233,0.7848,0.0194
svm,0.6677,0.0305,0.6698,0.0304,0.6820,0.0240


In [4]:
# Gráfico comparativo de F1-macro na CV
fig, ax = plt.subplots(figsize=(8, 4))
modelos = df_cv.index.tolist()
f1_medias = df_cv["f1_macro_media"].values
f1_stds = df_cv["f1_macro_std"].values

bars = ax.barh(modelos, f1_medias, xerr=f1_stds, color="steelblue", alpha=0.8, capsize=5)
ax.set_xlabel("F1-macro (média ± std)")
ax.set_title("Validação Cruzada — F1-macro por Modelo (Baseline)")
ax.set_xlim(0, 1.05)
for i, (v, e) in enumerate(zip(f1_medias, f1_stds)):
    ax.text(v + e + 0.01, i, f"{v:.3f}", va="center", fontsize=9)
fig.savefig(PROJECT_ROOT / "reports" / "figures" / "baseline_cv_f1_macro.png", dpi=150, bbox_inches="tight")
plt.close("all")

## 3. Otimização de hiperparâmetros — GridSearchCV (Baseline)

> **Nota:** Este GridSearch usa as grades originais da Fase 1 e serve como **ponto de referência** para o Algoritmo Genético do Notebook 02.

In [5]:
grid_searches = run_grid_search(pipelines, X_train, y_train)

print("\nMelhores hiperparâmetros (GridSearch baseline):")
summarize_best_params(grid_searches)

[GridSearch] logistic_regression: 5 combinacoes x 5 folds...
  Melhores params : {'model__C': 0.1}
  Melhor F1-macro : 0.6022
[GridSearch] random_forest: 27 combinacoes x 5 folds...


  Melhores params : {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__n_estimators': 100}
  Melhor F1-macro : 0.7929
[GridSearch] svm: 16 combinacoes x 5 folds...


  Melhores params : {'model__C': 100.0, 'model__gamma': 'scale'}
  Melhor F1-macro : 0.7012

Melhores hiperparâmetros (GridSearch baseline):


,melhor_f1_macro_cv,model__C,model__max_depth,model__min_samples_leaf,model__n_estimators,model__gamma
modelo,,,,,,
logistic_regression,0.6022,0.1,NaN,NaN,NaN,NaN
random_forest,0.7929,NaN,NaN,1.0,100.0,NaN
svm,0.7012,100.0,NaN,NaN,NaN,scale


## 4. Avaliação no conjunto de teste

In [6]:
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

resultados = {}
for nome, gs in grid_searches.items():
    metricas = evaluate_model(
        nome=nome,
        pipeline=gs.best_estimator_,
        X_test=X_test,
        y_test=y_test,
        plot_confusion=True,
        save_path=str(FIGURES_DIR / f"baseline_confusion_{nome}.png"),
    )
    metricas["best_params"] = gs.best_params_
    resultados[nome] = metricas


AVALIACAO — LOGISTIC_REGRESSION
              precision    recall  f1-score   support

    low risk       0.73      0.70      0.71        69
    mid risk       0.49      0.48      0.48        50
   high risk       0.72      0.79      0.76        39

    accuracy                           0.65       158
   macro avg       0.65      0.66      0.65       158
weighted avg       0.65      0.65      0.65       158

ROC-AUC macro OvR: 0.8211

AVALIACAO — RANDOM_FOREST
              precision    recall  f1-score   support

    low risk       0.91      0.88      0.90        69
    mid risk       0.86      0.88      0.87        50
   high risk       0.93      0.95      0.94        39

    accuracy                           0.90       158
   macro avg       0.90      0.90      0.90       158
weighted avg       0.90      0.90      0.90       158

ROC-AUC macro OvR: 0.9813

AVALIACAO — SVM
              precision    recall  f1-score   support

    low risk       0.75      0.78      0.77        69


In [7]:
# Tabela comparativa final
df_resultado = build_comparison_table(resultados)
print("\nComparativo Final — Conjunto de Teste:")
df_resultado.drop(columns=["best_params"], errors="ignore")


Comparativo Final — Conjunto de Teste:


,accuracy,f1_macro,recall_low_risk,recall_mid_risk,recall_high_risk,precision_macro,roc_auc_macro_ovr
random_forest,0.8987,0.9017,0.8841,0.88,0.9487,0.8994,0.9813
svm,0.7152,0.709,0.7826,0.56,0.7949,0.7082,0.8728
logistic_regression,0.6519,0.6507,0.6957,0.48,0.7949,0.646,0.8211


## 5. Feature Importance — Random Forest

In [8]:
rf_pipeline = grid_searches["random_forest"].best_estimator_

importancias = plot_feature_importance(
    pipeline=rf_pipeline,
    feature_names=X_train.columns.tolist(),
    save_path=str(FIGURES_DIR / "baseline_feature_importance_rf.png"),
)

print("\nFeature Importance (Random Forest):")
print(importancias)


Feature Importance (Random Forest):
BS             0.333689
SystolicBP     0.189115
Age            0.166647
DiastolicBP    0.126582
HeartRate      0.108789
BodyTemp       0.075178
dtype: float64


## 6. Salvar resultados baseline

Esses resultados serão usados como referência no Notebook 02 (Algoritmo Genético) para comparação antes/depois da otimização.

In [9]:
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"

# Remove best_params do JSON de métricas (não serializável de forma limpa)
baseline_export = {}
for nome, metricas in resultados.items():
    baseline_export[nome] = {k: v for k, v in metricas.items() if k != "best_params"}
    baseline_export[nome]["best_params"] = {
        k: str(v) for k, v in metricas.get("best_params", {}).items()
    }

output_path = EXPERIMENTS_DIR / "baseline_results.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(baseline_export, f, indent=2, ensure_ascii=False)

print(f"Resultados baseline salvos em: {output_path}")
print(json.dumps(baseline_export, indent=2, ensure_ascii=False))

Resultados baseline salvos em: /Users/igornatanael/workspace/tech-challenge-fiap-2/experiments/baseline_results.json
{
  "logistic_regression": {
    "accuracy": 0.6519,
    "f1_macro": 0.6507,
    "recall_low_risk": 0.6957,
    "recall_mid_risk": 0.48,
    "recall_high_risk": 0.7949,
    "precision_macro": 0.646,
    "roc_auc_macro_ovr": 0.8211,
    "best_params": {
      "model__C": "0.1"
    }
  },
  "random_forest": {
    "accuracy": 0.8987,
    "f1_macro": 0.9017,
    "recall_low_risk": 0.8841,
    "recall_mid_risk": 0.88,
    "recall_high_risk": 0.9487,
    "precision_macro": 0.8994,
    "roc_auc_macro_ovr": 0.9813,
    "best_params": {
      "model__max_depth": "None",
      "model__min_samples_leaf": "1",
      "model__n_estimators": "100"
    }
  },
  "svm": {
    "accuracy": 0.7152,
    "f1_macro": 0.709,
    "recall_low_risk": 0.7826,
    "recall_mid_risk": 0.56,
    "recall_high_risk": 0.7949,
    "precision_macro": 0.7082,
    "roc_auc_macro_ovr": 0.8728,
    "best_params"

## 7. Conclusão

| Modelo | F1-macro | ROC-AUC | Recall High Risk |
|--------|----------|---------|------------------|
| Random Forest | — | — | — |
| SVM | — | — | — |
| Regressão Logística | — | — | — |

> Preencher após execução. Esses valores são o **ponto de partida** — o Algoritmo Genético (Notebook 02) vai tentar superá-los automaticamente.